#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schemaSilver", "silver")
dbutils.widgets.text("catalogo", "project_dev")

In [0]:
schemaSilver = dbutils.widgets.get("schemaSilver")
catalogo = dbutils.widgets.get("catalogo")

# definir rutas
path_target = f"{catalogo}.{schemaSilver}"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, broadcast
from pyspark.sql import functions as F

##orders

In [0]:

df_orders = spark.table( f"{catalogo}.{schemaSilver}.orders_transform ")


##Product

In [0]:
df_product = (
    spark.table( f" {catalogo}.{schemaSilver}.products_transform ")
    .select( 
            F.col("product_id").alias("product_id_llave"),
            "product_category_name", 
            "product_category_name_english"
    )
)


## sellers

In [0]:
### leer el dataframe
df_sellers = (
    spark.table( f" {catalogo}.{schemaSilver}.sellers")
    .select( 
            F.col("seller_id").alias("seller_id_llave"),
            "seller_city", 
            "seller_state"
    )
)

## customers

In [0]:
### leer el dataframe
df_customers = (
    spark.table( f" {catalogo}.{schemaSilver}.customers")
    .select( 
            F.col("customer_id").alias("customer_id_llave"),
            "customer_city", 
            "customer_state"
    )
)

## join orders, product, sellers y customers

In [0]:
### realizar el join con las dos tablas
df_orders = (
    df_orders.alias("o").join( df_product.alias("p") , df_orders["product_id"] == df_product["product_id_llave"],how="inner")
    .join( broadcast( df_sellers.alias("s")) , df_orders["seller_id"] == df_sellers["seller_id_llave"],how="inner")
    .join( broadcast( df_customers.alias("c")) , df_orders["customer_id"] == df_customers["customer_id_llave"],how="inner")
    .drop("product_id_llave", "seller_id_llave", "customer_id_llave")
)

#df_orders_items.display()

In [0]:
## Unpivot tabla para que los dieferentes medios de pago esten en una columna
df_orders = df_orders.selectExpr(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "customer_id",
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "product_category_name",
    "product_category_name_english",
    "seller_city",
    "seller_state",
    "customer_city",
    "customer_state",
    """
    stack(5,
        'boleto', boleto_price_end, boleto_freight_end,
        'not_defined', not_defined_price_end, not_defined_freight_end,
        'credit_card', credit_card_price_end, credit_card_freight_end,
        'voucher', voucher_price_end, voucher_freight_end,
        'debit_card', debit_card_price_end, debit_card_freight_end
    ) as (payment_type, price, freight)
    """
).filter("""price is not null
     AND NOT (price = 0 AND freight = 0)    
""")

## cargar tabla

In [0]:
### guardar la tabla clean en el target
df_orders.write.mode("overwrite").insertInto(f"{path_target}.orders_transform_end")

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${schemaSilver}.orders_transform_end
ZORDER BY product_id, seller_id

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${schemaSilver}.orders_transform_end RETAIN 24 HOURS

In [0]:
%sql
--- describir la historia de la tabla
DESCRIBE HISTORY ${catalogo}.${schemaSilver}.orders_transform_end